# Adaptive SQS Visibility Timeout Controller
This notebook runs the simulator end to end on a standard CPU. Simulation results and optional AWS validation are kept explicitly separate.

In [ ]:
!pip -q install numpy pandas matplotlib seaborn pytest boto3

## Locate the project
If this notebook is inside the uploaded/cloned project, leave `PROJECT_DIR` unchanged. To clone, uncomment the two lines and set your repository URL.

In [ ]:
from pathlib import Path
# !git clone https://github.com/YOUR_ACCOUNT/adaptive-sqs-controller.git
# PROJECT_DIR = Path('/content/adaptive-sqs-controller')
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'src').exists():
    raise FileNotFoundError('Upload/clone the complete project and set PROJECT_DIR.')
%cd {PROJECT_DIR}

## Verify the implementation

In [ ]:
!python -m unittest discover -s tests -v

## Editable quick demonstration
The demo uses simulated time and finishes quickly. Increase messages/seeds only after checking this run.

In [ ]:
from src.experiment import run_experiments, generate_plots
from IPython.display import display, Image

demo_config = {
    'n_messages': 50, 'workers': 6, 'arrival_rate': 1.0,
    'seeds': [11, 22], 'safety_margin': 2.0, 'history_window': 30,
    'scenarios': {
        'mixed_no_failure': {'workload': 'mixed', 'failure': 'none'},
        'mixed_with_failures': {'workload': 'mixed', 'failure': 'medium'},
        'sudden_shift': {'workload': 'sudden_shift', 'failure': 'low'}
    }
}
attempts, runs, aggregates = run_experiments(demo_config, 'results/colab_demo')
generate_plots(attempts, runs, 'results/colab_demo/plots')
display(runs.round(3))
display(aggregates.filter(regex='scenario|policy_name|duplicate_processing_rate_mean|mean_failure_recovery_time_s_mean|mean_end_to_end_latency_s_mean|throughput_jobs_per_s_mean').round(3))

In [ ]:
plot_files = sorted(Path('results/colab_demo/plots').glob('*.png'))
for plot in plot_files:
    display(Image(filename=str(plot), width=850))

## Larger repeated experiment (run separately)
This covers all scenarios, margin/window sweeps, and five seeds. It may take several minutes.

In [ ]:
# Uncomment for the full experiment.
# !python -m src.experiment --config configs/full.json --output results/full --plots

## Optional Google Drive export
Mount only if you want to copy generated artifacts.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results/colab_demo '/content/drive/MyDrive/adaptive-sqs-results/'

## Optional real SQS validation (small and user-authorized only)
Create a dedicated Standard queue and least-privilege IAM principal first. Put secrets in Colab Secrets—never in this notebook. Running these cells performs real AWS API calls and a short real-time wait. The project is complete without them.

In [ ]:
# Explicitly opt in by changing this after reviewing README and docs/iam-policy.json.
RUN_AWS_VALIDATION = False
if RUN_AWS_VALIDATION:
    import os
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
    os.environ['AWS_REGION'] = userdata.get('AWS_REGION')
    os.environ['ADAPTIVE_SQS_TEST_QUEUE_URL'] = userdata.get('ADAPTIVE_SQS_TEST_QUEUE_URL')
    from src.aws_sqs_adapter import Boto3SQSAdapter
    adapter = Boto3SQSAdapter.from_environment()
    print(adapter.validate_one({'job_id': 'colab-validation-1', 'job_type': 'short', 'processing_duration': 0.5}, timeout=3))